# Cross-Method Comparative Analysis (LoRA vs QLoRA vs DoRA)

**Run this notebook after all 6+5 experiments are complete.**

Dependencies:
- `results/baseline/per_example_scores.json` (Member 1)
- `checkpoints/lora_rank{4,8,16}/per_example_scores.json` (Member 1)
- `checkpoints/qlora_rank{4,8,16}/per_example_scores.json` (Member 2)
- `checkpoints/dora_rank{4,8,16}/per_example_scores.json` (Member 2)

This notebook generates:
1. cross-method headline table (Table 1 extended)
2. Paired bootstrap statistical significance (Table 2 extended)
3. Rank scaling plot (LoRA / QLoRA / DoRA three curves)
4. efficiency Pareto plot (Accuracy vs Peak GPU Memory)
5. Loss curves comparison plot
6. Qualitative wins error analysis

## 0. Environment setup

In [ ]:
# Colab setup
import os, sys
if not os.path.exists('GR5293-peft-medical-vqa'):
    !git clone https://github.com/qiujunzhang03-7/GR5293-peft-medical-vqa.git
%cd GR5293-peft-medical-vqa
%pip install -q numpy pandas matplotlib seaborn rouge-score pyyaml

PROJECT_ROOT = '/content/GR5293-peft-medical-vqa'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# If checkpoints were previously backed up to Drive, copy them back
from pathlib import Path
drive_backup = Path('/content/drive/MyDrive/peft_medical_vqa/checkpoints')
if drive_backup.exists():
    !cp -r {drive_backup}/* checkpoints/ 2>/dev/null
    print(f"Restored checkpoints from Drive")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.evaluation.statistical_tests import (
    bootstrap_ci, paired_bootstrap_ci_diff, mcnemar_test
)

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 11

## 1. Load results for all methods

In [ ]:
def load_method(name, scores_path, metrics_path=None):
    """Load per-example scores and training metrics for one method."""
    with open(scores_path) as f:
        scores = json.load(f)
    metrics = None
    if metrics_path and Path(metrics_path).exists():
        with open(metrics_path) as f:
            metrics = json.load(f)
    return {'name': name, 'scores': scores, 'metrics': metrics}

methods = []

# Baseline (Member 1)
methods.append(load_method(
    'Baseline',
    'results/baseline/per_example_scores.json',
    'results/baseline/metrics.json'
))

# LoRA (Member 1)
for r in [4, 8, 16]:
    p = Path(f'checkpoints/lora_rank{r}')
    if (p / 'per_example_scores.json').exists():
        methods.append(load_method(f'LoRA r={r}', p/'per_example_scores.json', p/'training_metrics.json'))

# QLoRA (Member 2)
for r in [4, 8, 16]:
    p = Path(f'checkpoints/qlora_rank{r}')
    if (p / 'per_example_scores.json').exists():
        methods.append(load_method(f'QLoRA r={r}', p/'per_example_scores.json', p/'training_metrics.json'))

# DoRA (Member 2)
for r in [4, 8, 16]:
    p = Path(f'checkpoints/dora_rank{r}')
    if (p / 'per_example_scores.json').exists():
        methods.append(load_method(f'DoRA r={r}', p/'per_example_scores.json', p/'training_metrics.json'))

# Q-DoRA optional
qdora_path = Path('checkpoints/qdora_rank8')
if (qdora_path / 'per_example_scores.json').exists():
    methods.append(load_method('Q-DoRA r=8', qdora_path/'per_example_scores.json', qdora_path/'training_metrics.json'))

print(f"Loaded {len(methods)} methods:")
for m in methods:
    has_metrics = '✅' if m['metrics'] else '❌'
    print(f"  {has_metrics}  {m['name']}")

# Validate ID consistency
base_ids = methods[0]['scores']['ids']
for m in methods[1:]:
    assert m['scores']['ids'] == base_ids, f"❌ {m['name']} ids do not match!"
print(f"\n✅ All methods were evaluated on the same {len(base_ids)} test examples")

## 2. Headline Table (Table 1 extended)

In [ ]:
rows = []
qtypes = methods[0]['scores']['qtypes']
cl_idx = [i for i, q in enumerate(qtypes) if q == 'closed']
op_idx = [i for i, q in enumerate(qtypes) if q == 'open']

for m in methods:
    s = m['scores']
    
    # 95% bootstrap CI for each metric
    closed_correct = [s['correct'][i] for i in cl_idx]
    open_f1 = [s['f1'][i] for i in op_idx]
    
    closed_ci = bootstrap_ci(closed_correct, seed=42)
    open_ci = bootstrap_ci(open_f1, seed=42)
    overall_ci = bootstrap_ci(s['correct'], seed=42)
    
    # Training statistics (baseline does not have these)
    if m['metrics'] and 'params' in m['metrics']:
        trainable = m['metrics']['params']['trainable']
        trainable_pct = m['metrics']['params']['trainable_pct']
        peak_gpu = m['metrics']['peak_gpu_memory_gb']
        train_h = m['metrics']['training']['total_seconds'] / 3600
    else:
        trainable = 0
        trainable_pct = 0.0
        peak_gpu = None
        train_h = None
    
    rows.append({
        'Method': m['name'],
        'Trainable': f'{trainable:,}' if trainable else '0',
        'Trainable %': f'{trainable_pct:.4f}' if trainable_pct else '0.0000',
        'Closed EM': f"{closed_ci['point']:.4f} [{closed_ci['lower']:.4f}, {closed_ci['upper']:.4f}]",
        'Open Token-F1': f"{open_ci['point']:.4f} [{open_ci['lower']:.4f}, {open_ci['upper']:.4f}]",
        'Overall EM': f"{overall_ci['point']:.4f} [{overall_ci['lower']:.4f}, {overall_ci['upper']:.4f}]",
        'Peak GPU (GB)': f'{peak_gpu:.2f}' if peak_gpu else 'N/A',
        'Train (h)': f'{train_h:.2f}' if train_h else 'N/A',
    })

df = pd.DataFrame(rows)
print("=" * 120)
print("Table 1: Cross-method comparison (95% bootstrap CIs)")
print("=" * 120)
print(df.to_string(index=False))

# Save the markdown table
out_dir = Path('results/tables')
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / 'cross_method_results.md', 'w') as f:
    f.write("# Table 1: Cross-Method Results\n\n")
    f.write(df.to_markdown(index=False))
print(f"\n→ {out_dir / 'cross_method_results.md'}")

## 3. Significance Table (Table 2 extended) - Paired bootstrap + McNemar

In [ ]:
def compare(method_a, method_b):
    """Return all statistical differences for method_b - method_a."""
    a = method_a['scores']
    b = method_b['scores']
    
    # Overall EM
    overall = paired_bootstrap_ci_diff(a['correct'], b['correct'], seed=42)
    mc = mcnemar_test(a['correct'], b['correct'])
    
    # Closed EM
    a_c = [a['correct'][i] for i in cl_idx]
    b_c = [b['correct'][i] for i in cl_idx]
    closed = paired_bootstrap_ci_diff(a_c, b_c, seed=42)
    
    # Open Token-F1
    a_o = [a['f1'][i] for i in op_idx]
    b_o = [b['f1'][i] for i in op_idx]
    open_d = paired_bootstrap_ci_diff(a_o, b_o, seed=42)
    
    return {
        'closed': closed, 'open': open_d, 'overall': overall, 'mcnemar': mc
    }

def fmt_diff(d):
    sig = ''
    if d['p_two_sided'] < 0.001: sig = '***'
    elif d['p_two_sided'] < 0.01: sig = '**'
    elif d['p_two_sided'] < 0.05: sig = '*'
    return f"{d['point']:+.4f} [{d['lower']:+.4f}, {d['upper']:+.4f}] {sig}"

# Key comparison pairs
method_dict = {m['name']: m for m in methods}
comparisons = []

# 1) All PEFT methods vs Baseline (RQ1)
if 'Baseline' in method_dict:
    base = method_dict['Baseline']
    for name, m in method_dict.items():
        if name == 'Baseline': continue
        c = compare(base, m)
        comparisons.append({
            'Comparison': f'{name} − Baseline',
            'Closed EM Δ': fmt_diff(c['closed']),
            'Open F1 Δ': fmt_diff(c['open']),
            'Overall EM Δ': fmt_diff(c['overall']),
            'McNemar p': f"{c['mcnemar']['p_value']:.4g}",
        })

# 2) QLoRA vs LoRA at matched rank (RQ2)
for r in [4, 8, 16]:
    if f'LoRA r={r}' in method_dict and f'QLoRA r={r}' in method_dict:
        c = compare(method_dict[f'LoRA r={r}'], method_dict[f'QLoRA r={r}'])
        comparisons.append({
            'Comparison': f'QLoRA r={r} − LoRA r={r}',
            'Closed EM Δ': fmt_diff(c['closed']),
            'Open F1 Δ': fmt_diff(c['open']),
            'Overall EM Δ': fmt_diff(c['overall']),
            'McNemar p': f"{c['mcnemar']['p_value']:.4g}",
        })

# 3) DoRA vs LoRA at matched rank (RQ3)
for r in [4, 8, 16]:
    if f'LoRA r={r}' in method_dict and f'DoRA r={r}' in method_dict:
        c = compare(method_dict[f'LoRA r={r}'], method_dict[f'DoRA r={r}'])
        comparisons.append({
            'Comparison': f'DoRA r={r} − LoRA r={r}',
            'Closed EM Δ': fmt_diff(c['closed']),
            'Open F1 Δ': fmt_diff(c['open']),
            'Overall EM Δ': fmt_diff(c['overall']),
            'McNemar p': f"{c['mcnemar']['p_value']:.4g}",
        })

# 4) DoRA vs QLoRA at matched rank
for r in [4, 8, 16]:
    if f'QLoRA r={r}' in method_dict and f'DoRA r={r}' in method_dict:
        c = compare(method_dict[f'QLoRA r={r}'], method_dict[f'DoRA r={r}'])
        comparisons.append({
            'Comparison': f'DoRA r={r} − QLoRA r={r}',
            'Closed EM Δ': fmt_diff(c['closed']),
            'Open F1 Δ': fmt_diff(c['open']),
            'Overall EM Δ': fmt_diff(c['overall']),
            'McNemar p': f"{c['mcnemar']['p_value']:.4g}",
        })

df_sig = pd.DataFrame(comparisons)
print("Table 2: paired statistical significance (* p<0.05, ** p<0.01, *** p<0.001)")
print("=" * 120)
print(df_sig.to_string(index=False))

with open(out_dir / 'cross_method_significance.md', 'w') as f:
    f.write("# Table 2: Cross-Method Statistical Significance\n\n")
    f.write("Paired bootstrap CIs (10,000 resamples). \\* p<0.05, \\*\\* p<0.01, \\*\\*\\* p<0.001.\n\n")
    f.write(df_sig.to_markdown(index=False))
print(f"\n→ {out_dir / 'cross_method_significance.md'}")

## 4. Rank Scaling plot (LoRA vs QLoRA vs DoRA, three curves)

In [ ]:
fig_dir = Path('results/figures')
fig_dir.mkdir(parents=True, exist_ok=True)

# Collect (method, rank, metric) point estimates
rank_data = []
for m in methods:
    if m['name'] == 'Baseline': continue
    if 'r=' not in m['name']: continue
    
    method_type = m['name'].split(' ')[0]   # 'LoRA' / 'QLoRA' / 'DoRA' / 'Q-DoRA'
    rank = int(m['name'].split('=')[1])
    s = m['scores']
    
    rank_data.append({
        'method': method_type, 'rank': rank,
        'closed_em': np.mean([s['correct'][i] for i in cl_idx]),
        'open_f1': np.mean([s['f1'][i] for i in op_idx]),
        'overall_em': np.mean(s['correct']),
    })

df_rank = pd.DataFrame(rank_data)
print(df_rank)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
metric_names = ['closed_em', 'open_f1', 'overall_em']
metric_titles = ['Closed EM', 'Open Token-F1', 'Overall EM']
baseline_vals = {
    'closed_em': np.mean([methods[0]['scores']['correct'][i] for i in cl_idx]),
    'open_f1': np.mean([methods[0]['scores']['f1'][i] for i in op_idx]),
    'overall_em': np.mean(methods[0]['scores']['correct']),
}
colors = {'LoRA': '#1f77b4', 'QLoRA': '#ff7f0e', 'DoRA': '#2ca02c'}
markers = {'LoRA': 'o', 'QLoRA': 's', 'DoRA': '^'}

for ax, metric, title in zip(axes, metric_names, metric_titles):
    for method_type in ['LoRA', 'QLoRA', 'DoRA']:
        sub = df_rank[df_rank['method'] == method_type].sort_values('rank')
        if not sub.empty:
            ax.plot(sub['rank'], sub[metric],
                    marker=markers[method_type], color=colors[method_type],
                    linewidth=2, markersize=10, label=method_type)
    
    # baseline horizontal line
    ax.axhline(baseline_vals[metric], color='gray', linestyle='--', alpha=0.6, label='Baseline')
    
    ax.set_xlabel('LoRA Rank')
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.set_xticks([4, 8, 16])
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Rank Scaling: LoRA vs QLoRA vs DoRA on VQA-RAD', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(fig_dir / 'rank_scaling_all_methods.png', bbox_inches='tight')
plt.savefig(fig_dir / 'rank_scaling_all_methods.pdf', bbox_inches='tight')
plt.show()
print(f"→ {fig_dir / 'rank_scaling_all_methods.{png,pdf}'}")

## 5. efficiency Pareto plot (Accuracy vs Peak GPU Memory)

In [ ]:
pareto_data = []
for m in methods:
    if not m['metrics']: continue
    if 'params' not in m['metrics']: continue
    
    method_type = m['name'].split(' ')[0]
    rank = m['name'].split('=')[1] if 'r=' in m['name'] else None
    
    pareto_data.append({
        'name': m['name'],
        'method': method_type,
        'rank': rank,
        'overall_em': np.mean(m['scores']['correct']),
        'peak_gpu_gb': m['metrics']['peak_gpu_memory_gb'],
        'trainable_M': m['metrics']['params']['trainable'] / 1e6,
    })

df_pareto = pd.DataFrame(pareto_data)
print(df_pareto)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left plot: Accuracy vs Peak GPU
ax = axes[0]
for method_type in df_pareto['method'].unique():
    sub = df_pareto[df_pareto['method'] == method_type]
    color = colors.get(method_type, 'red')
    marker = markers.get(method_type, 'D')
    ax.scatter(sub['peak_gpu_gb'], sub['overall_em'],
               s=150, color=color, marker=marker,
               edgecolor='black', linewidth=0.8, label=method_type, alpha=0.85)
    for _, row in sub.iterrows():
        if row['rank']:
            ax.annotate(f"r={row['rank']}", (row['peak_gpu_gb'], row['overall_em']),
                        fontsize=8, ha='left', va='bottom', xytext=(5, 3),
                        textcoords='offset points')

ax.set_xlabel('Peak GPU Memory (GB)')
ax.set_ylabel('Overall EM')
ax.set_title('Accuracy-Memory Pareto Frontier')
ax.legend()
ax.grid(True, alpha=0.3)

# Right plot: Accuracy vs Trainable Params
ax = axes[1]
for method_type in df_pareto['method'].unique():
    sub = df_pareto[df_pareto['method'] == method_type]
    color = colors.get(method_type, 'red')
    marker = markers.get(method_type, 'D')
    ax.scatter(sub['trainable_M'], sub['overall_em'],
               s=150, color=color, marker=marker,
               edgecolor='black', linewidth=0.8, label=method_type, alpha=0.85)
    for _, row in sub.iterrows():
        if row['rank']:
            ax.annotate(f"r={row['rank']}", (row['trainable_M'], row['overall_em']),
                        fontsize=8, ha='left', va='bottom', xytext=(5, 3),
                        textcoords='offset points')

ax.set_xlabel('Trainable Parameters (M)')
ax.set_ylabel('Overall EM')
ax.set_title('Accuracy-Parameter Pareto')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(fig_dir / 'efficiency_pareto.png', bbox_inches='tight')
plt.savefig(fig_dir / 'efficiency_pareto.pdf', bbox_inches='tight')
plt.show()
print(f"→ {fig_dir / 'efficiency_pareto.{png,pdf}'}")

## 6. Loss Curves comparison plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for m in methods:
    if not m['metrics']: continue
    if 'training' not in m['metrics']: continue
    
    loss_curve = m['metrics']['training'].get('loss_curve', [])
    if not loss_curve: continue
    
    steps = [pt['step'] for pt in loss_curve]
    losses = [pt['loss'] for pt in loss_curve]
    
    method_type = m['name'].split(' ')[0]
    color = colors.get(method_type, 'gray')
    
    # rank=8 solid line, others dashed/dotted
    ls = '-' if 'r=8' in m['name'] else ('--' if 'r=4' in m['name'] else ':')
    
    ax.plot(steps, losses, color=color, linestyle=ls, linewidth=1.5, alpha=0.8, label=m['name'])

ax.set_xlabel('Optimizer Step')
ax.set_ylabel('Training Loss')
ax.set_title('Training Loss Curves: LoRA vs QLoRA vs DoRA')
ax.legend(loc='upper right', fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(fig_dir / 'loss_curves_all_methods.png', bbox_inches='tight')
plt.savefig(fig_dir / 'loss_curves_all_methods.pdf', bbox_inches='tight')
plt.show()
print(f"→ {fig_dir / 'loss_curves_all_methods.{png,pdf}'}")

## 7. Improvement Bar Chart

In [ ]:
if 'Baseline' in method_dict:
    base_correct = np.mean(method_dict['Baseline']['scores']['correct'])
    base_closed = np.mean([method_dict['Baseline']['scores']['correct'][i] for i in cl_idx])
    base_open = np.mean([method_dict['Baseline']['scores']['f1'][i] for i in op_idx])
    
    improve_data = []
    for name, m in method_dict.items():
        if name == 'Baseline': continue
        s = m['scores']
        improve_data.append({
            'method': name,
            'Closed EM Δ': np.mean([s['correct'][i] for i in cl_idx]) - base_closed,
            'Open F1 Δ':   np.mean([s['f1'][i] for i in op_idx]) - base_open,
            'Overall EM Δ': np.mean(s['correct']) - base_correct,
        })
    
    df_imp = pd.DataFrame(improve_data)
    
    fig, ax = plt.subplots(figsize=(13, 5))
    x = np.arange(len(df_imp))
    w = 0.27
    
    ax.bar(x - w, df_imp['Closed EM Δ'], w, label='Closed EM', color='#1f77b4')
    ax.bar(x,     df_imp['Open F1 Δ'], w, label='Open F1', color='#ff7f0e')
    ax.bar(x + w, df_imp['Overall EM Δ'], w, label='Overall EM', color='#2ca02c')
    
    ax.set_xticks(x)
    ax.set_xticklabels(df_imp['method'], rotation=30, ha='right')
    ax.set_ylabel('Improvement over Baseline (pp)')
    ax.set_title('PEFT Improvement over Zero-Shot Baseline')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(fig_dir / 'improvements_bar_all.png', bbox_inches='tight')
    plt.savefig(fig_dir / 'improvements_bar_all.pdf', bbox_inches='tight')
    plt.show()
    print(f"→ {fig_dir / 'improvements_bar_all.{png,pdf}'}")

## 8. Qualitative wins error analysis

Find representative examples that each method corrected relative to the baseline.

In [ ]:
# Load predictions for each method
def load_predictions(name, path):
    if not Path(path).exists(): return None
    return [json.loads(line) for line in open(path)]

preds_baseline = load_predictions('Baseline', 'results/baseline/predictions.jsonl')
if not preds_baseline:
    print("⚠️ baseline predictions do not exist")
else:
    err_dir = Path('results/error_analysis')
    err_dir.mkdir(parents=True, exist_ok=True)
    
    for name, m in method_dict.items():
        if name == 'Baseline': continue
        
        ckpt = name.lower().replace(' ', '_').replace('=', '')
        pred_file = Path(f'checkpoints/{ckpt}/lora_predictions.jsonl')
        if not pred_file.exists():
            continue
        
        preds = load_predictions(name, pred_file)
        if not preds: continue
        
        # Find cases where baseline is wrong and the method is correct
        wins = []
        for i in range(len(preds_baseline)):
            base_correct = method_dict['Baseline']['scores']['correct'][i]
            method_correct = m['scores']['correct'][i]
            if base_correct == 0 and method_correct == 1:
                wins.append({
                    'id': preds_baseline[i]['id'],
                    'qtype': preds_baseline[i]['qtype'],
                    'question': preds_baseline[i]['question'],
                    'reference': preds_baseline[i]['reference'],
                    'baseline_pred': preds_baseline[i]['prediction'],
                    'method_pred': preds[i]['prediction'],
                })
        
        n_closed = sum(1 for w in wins if w['qtype'] == 'closed')
        n_open = sum(1 for w in wins if w['qtype'] == 'open')
        print(f"{name}: {len(wins)} wins ({n_closed} closed, {n_open} open)")
        
        out = err_dir / f'baseline_vs_{ckpt}.json'
        with open(out, 'w') as f:
            json.dump({
                'method': name,
                'total_wins': len(wins),
                'closed_wins': n_closed,
                'open_wins': n_open,
                'examples': wins[:30],   # save the first 30
            }, f, indent=2)
        print(f"  → {out}")

## 9. Shared and unique wins across methods (advanced analysis)

Which examples did each method fix? A Venn-diagram-style comparison.

In [ ]:
# Use r=8 as the representative matched rank (matched rank)
if all(f'{m} r=8' in method_dict for m in ['LoRA', 'QLoRA', 'DoRA']):
    base_c = method_dict['Baseline']['scores']['correct']
    
    fixed_by = {}
    for m_name in ['LoRA r=8', 'QLoRA r=8', 'DoRA r=8']:
        method_c = method_dict[m_name]['scores']['correct']
        fixed_by[m_name] = set(
            i for i in range(len(base_c))
            if base_c[i] == 0 and method_c[i] == 1
        )
    
    # fixed by all three methods
    all_three = fixed_by['LoRA r=8'] & fixed_by['QLoRA r=8'] & fixed_by['DoRA r=8']
    
    # unique fixes
    only_lora = fixed_by['LoRA r=8'] - fixed_by['QLoRA r=8'] - fixed_by['DoRA r=8']
    only_qlora = fixed_by['QLoRA r=8'] - fixed_by['LoRA r=8'] - fixed_by['DoRA r=8']
    only_dora = fixed_by['DoRA r=8'] - fixed_by['LoRA r=8'] - fixed_by['QLoRA r=8']
    
    print("=" * 60)
    print("r=8 Win distribution across the three methods (vs Baseline)")
    print("=" * 60)
    print(f"  LoRA r=8 fixed:        {len(fixed_by['LoRA r=8'])}")
    print(f"  QLoRA r=8 fixed:       {len(fixed_by['QLoRA r=8'])}")
    print(f"  DoRA r=8 fixed:        {len(fixed_by['DoRA r=8'])}")
    print(f"  fixed by all three:        {len(all_three)}")
    print(f"  fixed only by LoRA:         {len(only_lora)}")
    print(f"  fixed only by QLoRA:        {len(only_qlora)}")
    print(f"  fixed only by DoRA:         {len(only_dora)}")
    
    # Draw the Venn diagram using matplotlib_venn
    try:
        from matplotlib_venn import venn3
        fig, ax = plt.subplots(figsize=(8, 7))
        venn3([fixed_by['LoRA r=8'], fixed_by['QLoRA r=8'], fixed_by['DoRA r=8']],
              set_labels=('LoRA r=8', 'QLoRA r=8', 'DoRA r=8'), ax=ax)
        ax.set_title('Examples Fixed by Each Method (vs Baseline)')
        plt.savefig(fig_dir / 'wins_venn_r8.png', bbox_inches='tight')
        plt.savefig(fig_dir / 'wins_venn_r8.pdf', bbox_inches='tight')
        plt.show()
    except ImportError:
        !pip install -q matplotlib_venn
        from matplotlib_venn import venn3
        fig, ax = plt.subplots(figsize=(8, 7))
        venn3([fixed_by['LoRA r=8'], fixed_by['QLoRA r=8'], fixed_by['DoRA r=8']],
              set_labels=('LoRA r=8', 'QLoRA r=8', 'DoRA r=8'), ax=ax)
        ax.set_title('Examples Fixed by Each Method (vs Baseline)')
        plt.savefig(fig_dir / 'wins_venn_r8.png', bbox_inches='tight')
        plt.show()

## 10. Generate summary report

In [ ]:
# Generate markdown summary
summary_lines = [
    "# Cross-Method Analysis Summary",
    "",
    "## Overview",
    f"- Evaluated {len(methods)} methods",
    f"- Test-set size: {len(base_ids)} (Closed={len(cl_idx)}, Open={len(op_idx)})",
    "",
    "## Key Findings",
]

# Find the best method for each metric
best_overall_em = max(methods, key=lambda m: np.mean(m['scores']['correct']))
best_closed = max(methods, key=lambda m: np.mean([m['scores']['correct'][i] for i in cl_idx]))
best_open = max(methods, key=lambda m: np.mean([m['scores']['f1'][i] for i in op_idx]))

summary_lines += [
    f"- **Best Overall EM:** {best_overall_em['name']} ({np.mean(best_overall_em['scores']['correct']):.4f})",
    f"- **Best Closed EM:** {best_closed['name']} ({np.mean([best_closed['scores']['correct'][i] for i in cl_idx]):.4f})",
    f"- **Best Open Token-F1:** {best_open['name']} ({np.mean([best_open['scores']['f1'][i] for i in op_idx]):.4f})",
    "",
]

# Most efficient
if pareto_data:
    most_efficient = min(pareto_data, key=lambda d: d['peak_gpu_gb'])
    summary_lines += [
        f"- **Lowest GPU memory:** {most_efficient['name']} ({most_efficient['peak_gpu_gb']:.2f} GB)",
        "",
    ]

summary_lines += [
    "## Output Files",
    "",
    "### Tables",
    "- `results/tables/cross_method_results.md` - Headline Table 1",
    "- `results/tables/cross_method_significance.md` - Significance Table 2",
    "",
    "### Figures",
    "- `results/figures/rank_scaling_all_methods.{png,pdf}`",
    "- `results/figures/efficiency_pareto.{png,pdf}`",
    "- `results/figures/loss_curves_all_methods.{png,pdf}`",
    "- `results/figures/improvements_bar_all.{png,pdf}`",
    "- `results/figures/wins_venn_r8.{png,pdf}`",
    "",
    "### Error Analysis",
    "- `results/error_analysis/baseline_vs_*.json`",
]

summary = '\n'.join(summary_lines)
with open('results/CROSS_METHOD_SUMMARY.md', 'w') as f:
    f.write(summary)
print(summary)
print(f"\n→ results/CROSS_METHOD_SUMMARY.md")